# AI2FEA 辅助智能体 - 演示 Notebook

## 项目简介

本 Notebook 演示如何直接调用 Abaqus 工具函数，无需通过 AI 智能体。
适用于：
- 快速测试 Abaqus 工具函数
- 调试仿真流程
- 学习工具函数的使用方法

## 环境设置

导入必要的库和工具函数

In [ ]:
# 导入必要的库
import os
import subprocess
import shutil
import importlib
import sys

# 添加配置文件路径
sys.path.insert(0, 'config_files')

# 导入 Abaqus 工具函数
import FEA_tools.tools as tools
importlib.reload(tools)  # 重新加载模块，确保使用最新代码

from FEA_tools.tools import (
    generate_input_file,              # 生成 Abaqus 输入文件
    run_abaqus,                        # 运行 Abaqus 作业
    extract_von_mises_stress_from_ODB  # 从 ODB 提取应力
)

print("✅ 工具函数导入成功！")

## 步骤 1：生成 Abaqus 输入文件

使用 `generate_input_file()` 函数生成带有指定位移的 Abaqus 输入文件。

**参数说明：**
- `applied_displacement`：施加的位移（单位：米）
- 示例：0.01 表示向下位移 0.01 米（10 毫米）

**注意：** 位移不应超过 0.2 米

In [ ]:
# 生成输入文件，位移为 0.01 米（10 毫米）
result = generate_input_file(0.02)
print(result)

In [ ]:
# 运行 Abaqus 作业
result = run_abaqus()
print(result)

In [ ]:
# 提取 Von-Mises 应力
vm_stress = extract_von_mises_stress_from_ODB()
print(f"提取结果：{vm_stress}")

In [ ]:
# 直接读取应力文件（验证结果）
stress_file = "FEA_Results/max_vm_stress.txt"
if os.path.exists(stress_file):
    with open(stress_file, 'r') as f:
        content = f.read().strip()
    # 解析：Maximum Mises stress in the cantilever beam is 68266728.000000
    if "is" in content:
        stress_value = float(content.split("is")[-1].strip())
        data = round(stress_value / 1e6, 2)
        print(f"最大 Von-Mises 应力：{data} MPa")
else:
    print("错误：应力文件不存在")

## 总结

本 Notebook 演示了完整的 Abaqus 仿真工作流：

1. ✅ **生成输入文件**：`generate_input_file(0.01)` - 创建带有 10mm 位移的模型
2. ✅ **运行仿真**：`run_abaqus()` - 执行 Abaqus 求解器
3. ✅ **提取应力**：`extract_von_mises_stress_from_ODB()` - 从 ODB 获取最大应力

### 与 AI 智能体的区别

- **Notebook 方式**：手动调用每个函数，适合调试和学习
- **AI 智能体方式**：通过自然语言描述任务，智能体自动选择和调用工具

### 下一步

- 尝试不同的位移值（如 0.02, 0.05, 0.1 米）
- 观察应力如何随位移变化
- 使用 AI 智能体自动化参数化研究（运行 `streamlit run main.py`）

### 文件位置

所有生成的文件都在 `FEA_Results/` 目录中：
- `cantilever_beam.inp` - 输入文件
- `cantilever_beam.odb` - 输出数据库
- `max_vm_stress.txt` - 应力结果